# Clase 057 — Fine-tuning: grid search y randomized search

Dejamos de tunear "a ojo": `GridSearchCV` para espacios chicos y discretos,
`RandomizedSearchCV` con distribuciones para espacios grandes o continuos, todo dentro de un
`Pipeline` para evitar leakage del preprocesamiento.

Requiere: `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from scipy.stats import randint, loguniform
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error

np.random.seed(42)
X, y = load_diabetes(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print('train', Xtr.shape, 'test', Xte.shape)

## 1. Baseline sin tunear

RandomForest con defaults: el número a batir.

In [ ]:
base = RandomForestRegressor(random_state=42, n_jobs=-1).fit(Xtr, ytr)
rmse_base = root_mean_squared_error(yte, base.predict(Xte))
print(f'RMSE test baseline: {rmse_base:.2f}')

## 2. GridSearchCV: producto cartesiano exhaustivo

Grid chico (≤9 combos) sobre `n_estimators` y `max_features`. `scoring` negado porque
sklearn siempre maximiza.

In [ ]:
grid = {'n_estimators': [100, 200], 'max_features': [0.4, 0.7, 1.0]}
t0 = time.perf_counter()
gs = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), grid,
                  cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gs.fit(Xtr, ytr)
t_grid = time.perf_counter() - t0
rmse_grid = root_mean_squared_error(yte, gs.best_estimator_.predict(Xte))
print('best_params:', gs.best_params_)
print(f'RMSE test grid: {rmse_grid:.2f}  ({t_grid:.1f}s, {len(gs.cv_results_["params"])} combos)')

## 3. RandomizedSearchCV con distribuciones

Muestrea `n_iter` puntos de distribuciones (`randint`, `loguniform`). Gana cuando pocos
hiperparámetros dominan o el espacio es grande.

In [ ]:
dist = {'n_estimators': randint(50, 400), 'max_features': loguniform(0.2, 1.0),
        'min_samples_leaf': randint(1, 20)}
t0 = time.perf_counter()
rs = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), dist,
                        n_iter=25, cv=5, scoring='neg_root_mean_squared_error',
                        n_jobs=-1, random_state=42)
rs.fit(Xtr, ytr)
t_rand = time.perf_counter() - t0
rmse_rand = root_mean_squared_error(yte, rs.best_estimator_.predict(Xte))
print('best_params:', {k: (round(v, 3) if isinstance(v, float) else v)
                       for k, v in rs.best_params_.items()})
print(f'RMSE test random: {rmse_rand:.2f}  ({t_rand:.1f}s, 25 trials)')

## 4. Pipeline + HPO: tunear preprocesamiento y modelo juntos

Con `Pipeline([('scaler', StandardScaler()), ('svr', SVR())])` tuneamos `svr__C` y
`svr__gamma`. El scaler queda DENTRO del CV: fitear el scaler afuera sería data leakage.

In [ ]:
pipe = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
pdist = {'svr__C': loguniform(1e0, 1e3), 'svr__gamma': loguniform(1e-4, 1e-1)}
svr_search = RandomizedSearchCV(pipe, pdist, n_iter=25, cv=5,
                                scoring='neg_root_mean_squared_error',
                                n_jobs=-1, random_state=42)
svr_search.fit(Xtr, ytr)
rmse_svr = root_mean_squared_error(yte, svr_search.best_estimator_.predict(Xte))
print('best_params:', {k: round(v, 5) for k, v in svr_search.best_params_.items()})
print(f'RMSE test SVR+pipeline: {rmse_svr:.2f}')
print('el scaler se fiteo por fold: sin leakage')

## 5. Inspección de cv_results_ + comparativa

In [ ]:
cvr = pd.DataFrame(rs.cv_results_)
cvr['rmse'] = -cvr['mean_test_score']
top = cvr.sort_values('rmse').head()[['param_n_estimators', 'param_min_samples_leaf', 'rmse']]
print('top-5 trials del RandomizedSearch:')
print(top.to_string(index=False))

comp = pd.DataFrame({
    'metodo': ['baseline', 'GridSearch', 'RandomSearch', 'SVR+pipe'],
    'RMSE_test': [rmse_base, rmse_grid, rmse_rand, rmse_svr],
    'segundos': [0.0, t_grid, t_rand, np.nan]}).round(2)
print('\n', comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(cvr['param_n_estimators'], cvr['rmse'], c='#37a')
ax.set_xlabel('n_estimators'); ax.set_ylabel('RMSE CV')
ax.set_title('cv_results_: RMSE vs n_estimators (hay meseta?)')
plt.tight_layout(); plt.show()

## Ejercicios

1. Ampliá el `param_grid` del ejercicio 2 agregando `max_depth ∈ {None, 10, 20}` y medí
   cuánto crece el número de fits (∏ |valores| × cv). ¿Sigue siendo viable con Grid?
2. Corré `RandomizedSearchCV` con `n_iter` en {10, 30, 60} y graficá `best_score_` vs n_iter.
   ¿Cuándo deja de mejorar?
3. Provocá el `ValueError: Invalid parameter 'C' for estimator Pipeline` pasando `'C'` en vez
   de `'svr__C'`, y arreglalo. ¿Por qué hace falta el doble underscore?
4. Reemplazá el scoring por `'r2'`. ¿Cambia el `best_params_`? ¿Qué implica sobre elegir la
   métrica antes de tunear?

## Conclusiones

- Grid es exhaustivo pero explota combinatorialmente; Random escala en espacios grandes.
- `loguniform` es clave para hiperparámetros en escala log (`C`, `gamma`, `learning_rate`).
- Meter el scaler en el `Pipeline` evita leakage: se fitea dentro de cada fold del CV.
- `cv_results_` te dice si conviene refinar el grid o si ya hay meseta.

## ✅ Soluciones de los ejercicios

Ejercicios del README resueltos sobre `load_diabetes` (offline; el README los plantea en California Housing, misma mecánica). Usamos `n_jobs=1`. El ejercicio 5 usa Optuna si está instalado y si no cae en un `RandomizedSearchCV` equivalente.

**Ej. 1 — GridSearchCV sobre RandomForest.** Grilla `n_estimators ∈ {50,100,200}` × `max_features ∈ {4,6,8}`, `cv=5`, scoring RMSE. Reportamos `best_params_` y RMSE en test.

In [ ]:

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error

X, y = load_diabetes(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
grid = {"n_estimators": [50, 100, 200], "max_features": [4, 6, 8]}
gs = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=1), grid,
                  cv=5, scoring="neg_root_mean_squared_error", n_jobs=1).fit(Xtr, ytr)
rmse_grid = root_mean_squared_error(yte, gs.best_estimator_.predict(Xte))
print("best_params:", gs.best_params_)
print(f"RMSE test (grid): {rmse_grid:.2f}  | {len(gs.cv_results_['params'])} combos evaluados")
assert rmse_grid > 0

**Ej. 2 — RandomizedSearchCV con distribuciones.** `randint`/`loguniform` en vez de grilla discreta; `n_iter=30`. Comparamos tiempo y score contra el grid del ej. 1.

In [ ]:

import time
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error

dist = {"n_estimators": randint(50, 200), "max_features": randint(2, 9),
        "min_samples_leaf": randint(1, 20)}
t0 = time.perf_counter()
rs = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=1), dist,
                        n_iter=15, cv=5, scoring="neg_root_mean_squared_error",
                        n_jobs=1, random_state=42).fit(Xtr, ytr)
t_rand = time.perf_counter() - t0
rmse_rand = root_mean_squared_error(yte, rs.best_estimator_.predict(Xte))
print("best_params:", rs.best_params_)
print(f"RMSE test (random, 15 trials): {rmse_rand:.2f}  ({t_rand:.1f}s)")
print("random explora rangos continuos que la grilla fija nunca visitaria")
assert rmse_rand > 0

**Ej. 3 — Pipeline + HPO (sin leakage).** Tuneamos `SVR` con `StandardScaler` dentro del `Pipeline`. Si escaláramos afuera del CV, cada fold de validación 'vería' la media/desvío del train completo: eso es data leakage.

In [ ]:

from scipy.stats import loguniform
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error

pipe = Pipeline([("scaler", StandardScaler()), ("svr", SVR())])
pdist = {"svr__C": loguniform(1e-1, 1e3), "svr__gamma": loguniform(1e-4, 1e-1)}
search = RandomizedSearchCV(pipe, pdist, n_iter=12, cv=5,
                            scoring="neg_root_mean_squared_error",
                            n_jobs=1, random_state=42).fit(Xtr, ytr)
rmse_svr = root_mean_squared_error(yte, search.best_estimator_.predict(Xte))
print("best_params:", {k: round(v, 5) for k, v in search.best_params_.items()})
print(f"RMSE test (SVR+pipeline): {rmse_svr:.2f}")
print("el StandardScaler se re-fitea dentro de cada fold -> estimacion sin fuga")

**Ej. 4 — Inspección de `cv_results_`.** Volcamos los resultados a un DataFrame, ordenamos por score y graficamos RMSE vs `n_estimators` para ver si hay meseta (subir árboles ya no rinde).

In [ ]:

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

cvr = pd.DataFrame(rs.cv_results_)
cvr["rmse"] = -cvr["mean_test_score"]
top = cvr.sort_values("rmse").head()[["param_n_estimators", "param_min_samples_leaf", "rmse"]]
print("top-5 configuraciones:")
print(top.to_string(index=False))
plt.scatter(cvr["param_n_estimators"].astype(int), cvr["rmse"], c="#37a")
plt.xlabel("n_estimators"); plt.ylabel("RMSE CV"); plt.title("RMSE vs n_estimators (meseta?)"); plt.show()
print("si la nube se aplana pasado cierto n_estimators, subir mas arboles no mejora")

**Ej. 5 — Optuna vs RandomizedSearch.** Con Optuna (TPE + MedianPruner) sobre el mismo RF; si Optuna no está instalado, comparamos contra el `RandomizedSearchCV` como búsqueda 'inteligente' de referencia.

In [ ]:

import numpy as np, time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        params = dict(n_estimators=trial.suggest_int("n_estimators", 50, 500),
                      max_features=trial.suggest_int("max_features", 2, 8),
                      min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20))
        m = RandomForestRegressor(random_state=42, n_jobs=1, **params)
        return -cross_val_score(m, Xtr, ytr, cv=5,
                                scoring="neg_root_mean_squared_error").mean()
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    t0 = time.perf_counter(); study.optimize(objective, n_trials=30); t_opt = time.perf_counter() - t0
    best = RandomForestRegressor(random_state=42, n_jobs=1, **study.best_params).fit(Xtr, ytr)
    rmse_opt = root_mean_squared_error(yte, best.predict(Xte))
    print(f"Optuna best params: {study.best_params}")
    print(f"RMSE test (Optuna, 30 trials): {rmse_opt:.2f}  ({t_opt:.1f}s)")
    assert rmse_opt > 0
except ImportError:
    print("optuna no instalado (`pip install optuna`).")
    print(f"Como referencia, el RandomizedSearchCV del ej.2 logro RMSE test = {rmse_rand:.2f}.")
    print("Optuna (TPE) suele igualar o superar ese numero con igual o menor presupuesto,")
    print("porque el sampler bayesiano concentra los trials donde el score viene mejorando.")